# Exploratory Analysis — Concord–Manchester Public Safety GIS

> **This is a portfolio demonstration using public or simulated data. It is not an official emergency management product and should not be used for operational decision-making.**

This notebook explores the cleaned and analyzed outputs of the pipeline. Run the scripts first:

```bash
python scripts/01_create_project_structure.py
python scripts/02_load_clean_data.py
python scripts/03_validate_gis_data.py
python scripts/04_generate_analysis_layers.py
python scripts/05_export_dashboard_tables.py
python scripts/06_generate_summary_report.py
```

Only `pandas` is required; `matplotlib` is optional (charts are skipped if it is not installed).

In [ ]:
import os
import pandas as pd

# Resolve the repo root whether the notebook runs from /notebooks or the root.
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
PROC = os.path.join(ROOT, 'data', 'processed')
DASH = os.path.join(ROOT, 'outputs', 'dashboard_tables')

try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except Exception:
    HAS_PLT = False
    print('matplotlib not installed - charts will be skipped (tables still print).')

## 1. Load the cleaned datasets

In [ ]:
incidents = pd.read_csv(os.path.join(PROC, 'incidents.csv'))
facilities = pd.read_csv(os.path.join(PROC, 'emergency_facilities.csv'))
shelters = pd.read_csv(os.path.join(PROC, 'shelters.csv'))
hospitals = pd.read_csv(os.path.join(PROC, 'hospitals.csv'))
towns = pd.read_csv(os.path.join(PROC, 'municipalities.csv'))

print(f'incidents:  {len(incidents)} rows')
print(f'facilities: {len(facilities)} rows')
print(f'shelters:   {len(shelters)} rows')
print(f'hospitals:  {len(hospitals)} rows')
print(f'towns:      {len(towns)} rows')
incidents.head()

## 2. Incidents by town

In [ ]:
by_town = incidents.groupby('town').size().sort_values(ascending=False)
print(by_town)

if HAS_PLT:
    ax = by_town.plot(kind='bar', title='Simulated incidents by town', figsize=(9, 4))
    ax.set_ylabel('incident count')
    plt.tight_layout()
    plt.show()

## 3. Incident severity & status

In [ ]:
print(pd.crosstab(incidents['severity'], incidents['status'], margins=True))

open_high = incidents[(incidents['status'].isin(['Open', 'In Progress'])) &
                      (incidents['severity'].isin(['High', 'Critical']))]
print(f'\nOpen high-severity incidents (act-now list): {len(open_high)}')
open_high[['incident_id', 'incident_type', 'severity', 'town', 'status']].head(15)

## 4. Analysis layer: priority & proximity

In [ ]:
prox = pd.read_csv(os.path.join(DASH, 'incident_facility_proximity.csv'))
print('Priority breakdown:')
print(prox['priority_category'].value_counts().sort_index())

coverage_gaps = prox[(prox['priority_category'].isin(['P1 - Immediate', 'P2 - Urgent'])) &
                     (prox['nearest_facility_miles'] > 3.0)]
print(f'\nCoverage-gap incidents (P1/P2 & >3 mi to facility): {len(coverage_gaps)}')
coverage_gaps.sort_values('nearest_facility_miles', ascending=False).head(10)

## 5. Town readiness summary

In [ ]:
town_summary = pd.read_csv(os.path.join(DASH, 'town_readiness_summary.csv'))
town_summary

## 6. Data quality results

In [ ]:
qa = pd.read_csv(os.path.join(DASH, 'qa_results.csv'))
print('Issues by severity:')
print(qa['severity'].value_counts())
print('\nIssues by check:')
print(qa['check_id'].value_counts().sort_index())
qa

## Takeaways

- Incident load concentrates in the larger towns (Manchester, Concord).
- A handful of high-priority incidents sit far from the nearest facility — coverage gaps worth planning attention.
- The validator flags the deliberately seeded data-quality defects, which would be fixed at the source before any operational use.

See `outputs/reports/summary_report.md` and `outputs/reports/qa_qc_report.md` for the generated reports.